In [87]:
import json
import pandas as pd
import pickle

We start by processing the card list.

In [116]:
with open('all_cards.pkl', 'rb') as f:
  all_cards = pickle.load(f)
df_cards = pd.DataFrame(all_cards)
print(f'Loaded {df_cards.shape[0]} cards')

Loaded 265 cards


On `arkhamdb`, a large number of decklists only have cards from the Core or Revised Core set, since that is the entry point into the game. The next largest group of decklists selects cards from the Core and Dunwich Legacy expansion, followed by decklists that go up to Path to Carcosa, etc., following the chronological order the expansions were released.

Because of this, if we tallied up all the cards used in all the decklists, the count will be biased toward the Core set cards and away from cards from the more recent expansions. But just because the most common gun used by Guardians is the .45 Automatic doesn't mean it's the best gun for each of Guardian investigator.

`rangersdb` decks follow this pattern so far, with most decks using just the Core set followed by decks that also draw from Stewards of the Valley. To account for this bias, here we define the order of Earthborne Ranger packs, which is what `rangersdb` calls them, and create some helper dictionaries.

In [117]:
PACK_ORDER = [
  'core',  # Core set
  'sotv',  # Stewards of the Valley
  'loa'    # Legacy of the Ancestors
]
PACK_TO_INDEX = {pack_id: i for i, pack_id in enumerate(PACK_ORDER)}
INDEX_TO_PACK = {i: pack_id for i, pack_id in enumerate(PACK_ORDER)}
df_cards["card_pack_index"] = df_cards["pack_id"].map(PACK_TO_INDEX)
card_id_to_pack_index = (df_cards
  .set_index("id")
  .to_dict()["card_pack_index"])
card_id_to_name = (df_cards
  .set_index("id")
  .to_dict()["name"])
print('Count cards in each pack:')
df_cards.groupby("card_pack_index").size()

Count cards in each pack:


card_pack_index
0    155
1     81
2     29
dtype: int64

Note that this reports 155 cards in the Core set when it actually contains 140. This is because it contains new versions of cards from [The Elder's Book of Uncommon Wisdom](https://thelivingvalley.earthbornegames.com/docs/updates/elders_book_of_uncommon_wisdom/), which is sometimes used to nerf overpowered cards. Many decks have not updated to the new version of cards, and maybe the player wouldn't include the newer version of the card. Here, we label each card `id` if it is or isn't their latest version.

In [118]:
# These cards id were enhanced in taboo set_01, so if the user liked the older
# version, they would also like the newer version
NOT_NERFED = ('01041', '01044', '01069', '01074', '01084')
# taboo_id is None for the original version of each card
df_cards["taboo_fill_id"] = df_cards["taboo_id"].fillna('set_00')
df_cards["latest_taboo_id"] = (df_cards
  .groupby("code")["taboo_fill_id"]
  .transform("max"))
df_cards["is_latest"] = (
  (df_cards["taboo_fill_id"] == df_cards["latest_taboo_id"])
  | df_cards["id"].isin(NOT_NERFED))

print('Latest cards in each pack:')
df_cards.groupby("card_pack_index").agg(
  latest=("is_latest", "sum"),
  not_latest=("is_latest", lambda x: (~x).sum()))

Latest cards in each pack:


,latest,not_latest
card_pack_index,,
0,145,10
1,81,0
2,29,0


Each deck has `set_type_id` and `set_id` attributes, but they are not quite the way we want to group cards:

In [119]:
(df_cards[["set_type_id", "set_id"]]
  .drop_duplicates()
  .sort_values(by=["set_type_id", "set_id"]))

,set_type_id,set_id
20,background,artisan
28,background,forager
10,background,shepherd
184,background,talespinner
11,background,traveler
8,specialty,artificer
1,specialty,conciliator
4,specialty,explorer
0,specialty,shaper
188,specialty,spirit_speaker


Players make eight different card selections during deck creation: their Role, the four Personality aspects, their Background cards, their Specialty cards, and their Outside Interest. So we modify `set_type_id` and `set_id` into the `supergroup_id` and `subgroup_id`, which groups cards for these eight choices.

In [120]:
def get_supergroup_id(card):
  if card["set_id"] == 'personality':
    return 'Personality'
  set_type_id = card["set_type_id"]
  if set_type_id == 'specialty' and card["type_id"] == 'role':
    return 'Role'
  return (set_type_id.capitalize()
          if isinstance(set_type_id, str) else set_type_id)

def get_subgroup_id(card):
  if card["set_id"] == "personality":
    return card["aspect_name"].title()
  return card["set_id"].replace('_', ' ').title()

df_cards["supergroup_id"] = df_cards.apply(get_supergroup_id, axis=1)
df_cards["subgroup_id"] = df_cards.apply(get_subgroup_id, axis=1)

print('Supergroup and subgroup combinations:')
(df_cards[["supergroup_id", "subgroup_id"]]
  .drop_duplicates()
  .sort_values(by=["supergroup_id", "subgroup_id"]))

Supergroup and subgroup combinations:


,supergroup_id,subgroup_id
20,Background,Artisan
28,Background,Forager
10,Background,Shepherd
184,Background,Talespinner
11,Background,Traveler
63,Personality,Awareness
58,Personality,Fitness
61,Personality,Focus
62,Personality,Spirit
54,Role,Artificer


That's all the preprocessing we need to do for the cards, now let's load the decks.

In [123]:
with open('decks.pkl', 'rb') as f:
  all_decks = pickle.load(f)
print(f'Loaded {len(all_decks)} decks between {min(all_decks.keys())} and {max(all_decks.keys())}')
decks_clean = [deck for deck in all_decks.values() if deck is not None]
df_all_decks = pd.DataFrame(decks_clean)
df_all_decks["taboo_set_id"] = df_all_decks["taboo_set_id"].fillna('set_00')
print(f'Loaded {df_all_decks.shape[0]} non-blank decks between {df_all_decks["id"].min()} annd {df_all_decks["id"].max()}')

Loaded 25146 decks between 1 and 25146
Loaded 14352 non-blank decks between 1 annd 25117


Each deck stores their Background, Specialty, and Role in the `meta` attribute. Here, we move them into their own columns, and we ignore some malformed decks that don't have all three.

In [124]:
def extract_background(meta):
  return meta.get("background", None)
def extract_specialty(meta):
  return meta.get("specialty", None)
def extract_role_id(meta):
  return meta.get("role", None)

df_all_decks["background"] = df_all_decks["meta"].apply(extract_background)
df_all_decks["specialty"] = df_all_decks["meta"].apply(extract_specialty)
df_all_decks["role_id"] = df_all_decks["meta"].apply(extract_role_id)

df_all_decks["is_ignore"] = (
  df_all_decks["background"].isna()
  | df_all_decks["specialty"].isna()
  | df_all_decks["role_id"].isna())
ignore_count = df_all_decks["is_ignore"].sum()
print(f'Ignored decks: {ignore_count}')

Ignored decks: 36


We want to look at decks that players made, so we ignore decks that are identical to the [premade decks](https://earthbornegames.com/resources/ranger-decks).

In [125]:
#| include: false
# ignore pre-built decks
def card_name_to_ids(name):
  return df_cards.loc[
    (df_cards["name"] == name),
    "id"
  ].tolist()

UNDAUNTED_SEEKER = {
  'background': 'traveler',
  'specialty': 'explorer',
  'role_id': card_name_to_ids('Undaunted Seeker'),
  'awa': 2,
  'spi': 2,
  'fit': 3,
  'foc': 1,
  'slots': (
    card_name_to_ids('Meticulous'),
    card_name_to_ids('Determined'),
    card_name_to_ids('Insightful'),
    card_name_to_ids('Persuasive'),
    card_name_to_ids('Eagle Eye'),
    card_name_to_ids('Ironwool Boots'),
    card_name_to_ids('Perfect Recall'),
    card_name_to_ids('Reverb Locket'),
    card_name_to_ids('Strider'),
    card_name_to_ids('Afforded by Nature'),
    card_name_to_ids('A Leaf in the Breeze'),
    card_name_to_ids('Boundary Sensor'),
    card_name_to_ids('Orlin Hiking Stave'),
    card_name_to_ids('Walk With Me'),
    card_name_to_ids('Trail Markers'),
  )
}

VOICE_OF_THE_ELDERS = {
  'background': 'shepherd',
  'specialty': 'conciliator',
  'role_id': card_name_to_ids('Voice of the Elders'),
  'awa': 1,
  'spi': 3,
  'fit': 2,
  'foc': 2,
  'slots': (
    card_name_to_ids('Engaging'),
    card_name_to_ids('Astute'),
    card_name_to_ids('Passionate'),
    card_name_to_ids('Perceptive'),
    card_name_to_ids('Calming Presence'),
    card_name_to_ids('Healing Touch'),
    card_name_to_ids('Homeward Bound'),
    card_name_to_ids('Oru the Sheep Dog'),
    card_name_to_ids('Paratrepsis Whistle'),
    card_name_to_ids('A Dear Friend'),
    card_name_to_ids('Follow in Footsteps'),
    card_name_to_ids('Intention Translator'),
    card_name_to_ids('Pokodo the Ferret'),
    card_name_to_ids('Surveyed Land'),
    card_name_to_ids('Energized Hiking Greaves'),
  )
}

MASTERFUL_ENGINEER = {
  'background': 'artisan',
  'specialty': 'artificer',
  'role_id': card_name_to_ids('Masterful Engineer'),
  'awa': 2,
  'spi': 2,
  'fit': 1,
  'foc': 3,
  'slots': (
    card_name_to_ids('Balanced'),
    card_name_to_ids('Inventive'),
    card_name_to_ids('Thorough'),
    card_name_to_ids('Compassionate'),
    card_name_to_ids('Favorite Gear'),
    card_name_to_ids('Functional Replica'),
    card_name_to_ids('Moment of Desperation'),
    card_name_to_ids('Pocketed Belt Pouch'),
    card_name_to_ids('Universal Power Cells'),
    card_name_to_ids('A Stone in the River'),
    card_name_to_ids('Ferinodex'),
    card_name_to_ids('Infusion Canteen'),
    card_name_to_ids('Memorill Sketchpad'),
    card_name_to_ids('Memlev Trekking Poles'),
    card_name_to_ids('Adaptable Multitool'),
  )
}

ADHERENT_OF_THE_FIRST_IDEAL = {
  'background': 'forager',
  'specialty': 'shaper',
  'role_id': card_name_to_ids('Adherent of the First Ideal'),
  'awa': 3,
  'spi': 1,
  'fit': 2,
  'foc': 2,
  'slots': (
    card_name_to_ids('Bold'),
    card_name_to_ids('Thoughtful'),
    card_name_to_ids('Versatile'),
    card_name_to_ids('Vigilant'),
    card_name_to_ids('Familiar Ground'),
    card_name_to_ids('Green Thumb'),
    card_name_to_ids('Local Fare'),
    card_name_to_ids('Loose-Leaf Tea Kit'),
    card_name_to_ids("Nature's Abundance"),
    card_name_to_ids('Root Snare'),
    card_name_to_ids('Shape the Earth'),
    card_name_to_ids('Sky Whip'),
    card_name_to_ids('Stave of the Sun'),
    card_name_to_ids('What Should Never Be'),
    card_name_to_ids('Harmonize'),
  )
}

CARTOGRAPHER_OF_MANY_WORLDS = {
  'background': 'talespinner',
  'specialty': 'spirit_speaker',
  'role_id': card_name_to_ids('Cartographer of Many Worlds'),
  'awa': 1,
  'spi': 2,
  'fit': 2,
  'foc': 3,
  'slots': (
    card_name_to_ids('Dauntless'),
    card_name_to_ids('Enthusiastic'),
    card_name_to_ids('Ingenious'),
    card_name_to_ids('Sensible'),
    card_name_to_ids('Wonders of the Valley'),
    card_name_to_ids('The Buck and the Wolhund'),
    card_name_to_ids('Book of Great Deeds'),
    card_name_to_ids('The Tireless Artisan'),
    card_name_to_ids('Songweave Mantle'),
    card_name_to_ids('Irix Spirit'),
    card_name_to_ids('The Path Opens Before Me'),
    card_name_to_ids('Atrox Spirit'),
    card_name_to_ids('The Long Walk'),
    card_name_to_ids('Lutrinal Spirit'),
    card_name_to_ids('Piercing Whistle'),
  )
}

def if_match_deck(deck, premade):
  for key, value in premade.items():
    if key == 'role_id':
      if deck['role_id'] not in value:
        return False

    elif key == 'slots':
      flatten_slots = [id for id_list in value for id in id_list]
      for card_id in deck['slots']:
        if card_id not in flatten_slots:
          return False

    elif deck[key] != value:
      return False

  return True

df_all_decks["is_ignore"] = (
  df_all_decks["is_ignore"]
  | df_all_decks.apply(lambda deck: if_match_deck(deck, UNDAUNTED_SEEKER), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, VOICE_OF_THE_ELDERS), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, MASTERFUL_ENGINEER), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, ADHERENT_OF_THE_FIRST_IDEAL), axis=1)
  | df_all_decks.apply(lambda deck: if_match_deck(deck, CARTOGRAPHER_OF_MANY_WORLDS), axis=1)
)
ignore_count = df_all_decks["is_ignore"].sum()
print(f'\nIgnored decks: {ignore_count}')


Ignored decks: 885


Now, we start analyzing the cards in each deck.

We assign each deck a `deck_pack_index` reflecting the highest `card_pack_index` among its cards. The reasoning here is, for example, if a deck contains a card from Stewards of the Valley, we assume the player considered all of the cards available up through Stewards of the Valley when choosing cards. So we want to identify and favor decks that consider more cards.

We also count the total equip value of the cards in the deck to get a sense of how much gear a typical deck has.

We also want to ignore decks that don't meed the following conditions:
- They have 15 cards
- None of the cards are spoilers, since cards with `spoiler` are cards like Rewards or Maladies that aren't included in starting decks

In [126]:
def check_cards(deck):
  slots = [card_id for card_id in deck["slots"].keys()]
  has_15_cards = (len(slots) == 15)
  # decks have a 16th card, their role card
  slots.append(deck["role_id"])
  cards = df_cards[df_cards["id"].isin(slots)]
  deck_pack_index = cards["card_pack_index"].max()
  equip_sum = cards["equip"].sum()
  has_spoiler = cards["spoiler"].any()
  is_latest = cards["is_latest"].all()
  return pd.Series({
    "deck_pack_index": deck_pack_index,
    "equip_sum": equip_sum,
    "has_15_cards": has_15_cards,
    "has_spoiler": has_spoiler,
    "is_latest": is_latest
  })

card_check = df_all_decks.apply(check_cards, axis=1)
# Drop columns so running this cell repeatedly doesn't throw an error
for col in card_check.columns:
  if col in df_all_decks.columns:
    print(f'Dropping column {col}')
    df_all_decks = df_all_decks.drop(columns=[col])
df_all_decks = pd.concat([df_all_decks, card_check], axis=1)
df_all_decks["deck_pack_index"] = (df_all_decks["deck_pack_index"]
  .astype('Int64'))

print(f"\ndeck_pack_index distribution:")
for deck_pack_index in range(len(PACK_ORDER)):
  pack_id = INDEX_TO_PACK[deck_pack_index]
  count = (df_all_decks["deck_pack_index"] == deck_pack_index).sum()
  print(f"  {pack_id} (index {deck_pack_index}): {count} decks")

df_all_decks["is_ignore"] = (
  df_all_decks["background"].isna()
  | df_all_decks["specialty"].isna()
  | df_all_decks["role_id"].isna()
  | ~df_all_decks["has_15_cards"]
  | df_all_decks["has_spoiler"]
  # | df_all_decks["is_latest"]
)
ignore_count = df_all_decks["is_ignore"].sum()
print(f'\nIgnored decks: {ignore_count}')


deck_pack_index distribution:
  core (index 0): 11034 decks
  sotv (index 1): 3130 decks
  loa (index 2): 184 decks

Ignored decks: 4618


Decks can have a lineage, having been derived from a `previous_deck` or `original_deck`. We will want to know the parent for all decks, even the ignored decks.

In [127]:
def get_all_decks_parent_id(all_decks_row):
  if all_decks_row["previous_deck"] is not None:
    previous_id = all_decks_row["previous_deck"]["id"]
    # Sometimes the previous deck has been deleted
    if previous_id in df_all_decks["id"].values:
      return previous_id

  if all_decks_row["original_deck"] is not None:
    original_id = all_decks_row["original_deck"]["deck"].get("id", None)
    if original_id in df_all_decks["id"].values:
      return original_id

  return None

df_all_decks["all_decks_parent_id"] = (df_all_decks
  .apply(get_all_decks_parent_id, axis=1)
  .astype('Int64'))
assert (set(df_all_decks["all_decks_parent_id"].dropna())
  <= set(df_all_decks["id"]))
num_parent = df_all_decks["all_decks_parent_id"].notna().sum()
num_all_decks = df_all_decks.shape[0]
print(f'Number of parent_id: {num_parent} out of {num_all_decks} decks')

Number of parent_id: 4468 out of 14352 decks


This new DataFrame only contains the non-ignored decks.

In [128]:
df_decks = df_all_decks[~df_all_decks["is_ignore"]].copy()
print(f'{df_decks.shape[0]} non-ignored decks')

9734 non-ignored decks


Continuing to analyze deck lineage, we also want to find the nearest ancestor that is not ignored, i.e., is also in `df_decks`.

In [129]:
def get_parent_id(deck):
  current_id = deck["all_decks_parent_id"]
  while pd.notna(current_id):
    if current_id in df_decks["id"].values:
      return current_id
      
    current_deck = df_all_decks[df_all_decks["id"] == current_id].iloc[0]
    current_id = current_deck["all_decks_parent_id"]

  return None

df_decks["parent_id"] = (df_decks
  .apply(get_parent_id, axis=1)
  .astype('Int64'))
assert set(df_decks["parent_id"].dropna()) <= set(df_decks["id"])
num_parent = df_decks["parent_id"].notna().sum()
num_decks = df_decks.shape[0]
print(f'Number of parent_id: {num_parent} out of {num_decks} decks')

Number of parent_id: 1262 out of 9734 decks


Later, we'll also use the nearest ancestor in `df_decks` that comes from a different user, which we'll call its `inspiration_id`.

In [130]:
def get_inspiration_id(deck):
  this_user_id = deck["user_id"]
  current_id = deck["parent_id"]
  while pd.notna(current_id):
    current_deck = df_decks[df_decks["id"] == current_id].iloc[0]
    current_user_id = current_deck["user_id"]
    if current_user_id != this_user_id:
      return current_id
      
    current_id = current_deck["parent_id"]

  return None

df_decks["inspiration_id"] = (df_decks
  .apply(get_inspiration_id, axis=1)
  .astype('Int64'))
num_inspiration = df_decks["inspiration_id"].notna().sum()
num_decks = df_decks.shape[0]
print(f'Number of inspiration_id: {num_inspiration} out of {num_decks} decks')

Number of inspiration_id: 672 out of 9734 decks


Now is where we get more subjective.
- For every `user_id`, first we ignore any deck `id` that is a `parent_id` for one of their own decks. The idea is the user has created an improved version of the parent deck, so we ignore the parent deck in favor of the child deck.
- Then, for each `["background", "specialty"]` combination the user made decks for, they are allocated 1 total `user_weight`, divided equally between those decks. So if the user made a bunch of very similar decks, those decks only get one vote split among all of them.

In [131]:
# Identify decks with non-zero user_weight
user_parent_ids = (df_decks.groupby("user_id")["parent_id"]
  .apply(lambda parents: set(parents.dropna())))
df_decks["user_weight"] = (df_decks
  .apply(lambda row: row["id"] not in user_parent_ids[row["user_id"]], axis=1)
  .astype('Int64'))

# Divide 1 user_weight among (user_id, background, specialty) decks
sum_weights = (df_decks
  .groupby(["user_id", "background", "specialty"])["user_weight"]
  .transform("sum"))
df_decks["user_weight"] = (df_decks["user_weight"]
  / sum_weights.where(sum_weights > 0, 1))

user_weight_sum = df_decks["user_weight"].sum()
user_weight_size = (df_decks["user_weight"] > 0).size
print(f'{user_weight_sum} total user_weight split among {user_weight_size} decks')

6832.0 total user_weight split among 9734 decks


Now, if a deck was inspired by another user's deck, the `user_weight` of the child deck is split between it and its inspiration. If the deck wasn't inspired by another user's deck, it keeps all of its `user_weight`. In this way, a higher `deck_weight` reflects more players who liked this deck and made their own versions.

In [132]:
# Decks with no inspiration keep their full user_weight
# Decks with inspiration donate half their user_weight
from_self = df_decks.apply(lambda row: 0.5 * row["user_weight"]
  if pd.notna(row["inspiration_id"]) else row["user_weight"], axis=1)

# Decks receive half the user_weight of decks they inspired
df_inspired = df_decks[df_decks["inspiration_id"].notna()]
sum_weights = df_inspired.groupby("inspiration_id")["user_weight"].sum()
from_inspired = df_decks["id"].map(lambda x: 0.5 * sum_weights.get(x, 0))

df_decks["deck_weight"] = from_self + from_inspired

deck_weight_sum = df_decks["deck_weight"].sum()
deck_weight_size = (df_decks["deck_weight"] > 0).size
print(f'{deck_weight_sum} total deck_weight split among {deck_weight_size} decks')

6832.0 total deck_weight split among 9734 decks


We will penalize decks that haven't adopted the taboo proportional to the decks who have adopted it.

In [133]:
taboo_counts = {taboo_id: (df_decks["taboo_set_id"] <= taboo_id).sum()
  for taboo_id in df_decks["taboo_set_id"].unique()}
max_taboo_count = max(taboo_counts.values())
map_taboo_weight = {taboo_id: float(count / max_taboo_count)
  for taboo_id, count in taboo_counts.items()}
df_decks["taboo_weight"] = df_decks["taboo_set_id"].map(map_taboo_weight)
print(f"Taboo weights: {map_taboo_weight}")
df_decks["group_score"] = df_decks["deck_weight"] * df_decks["taboo_weight"]

Taboo weights: {'set_00': 0.8914115471543045, 'set_01': 1.0}


Let's analyze which (Background, Specialty) combinations are the most popular(*). Each deck contributes its `deck_weight` to its (Background, Specialty) score.

However, this raw total score is pretty biased. The Core set has been around longer, so there has been more time to make decks using only the Core set. And as mentioned earlier, more players have access to the Core set than expansion packs.

So for each `pack_index`, we calculate its average score, as if we were dividing the total score awarded in that `pack_index` equally among the various options. Then for each option, we add up its score over the set of `pack_index` when it is available. We do the same for the average score. Comparing the option's total score to the average score gives us a popularity rating.

(*) For my arbitrary definition of popular.

In [134]:
def group_and_score(df, groupby_cols, score_col, pack_index_col,
    background=None, specialty=None):
  # filter by background and specialty if provided
  if background and specialty:
    df = df[
      (df["Background"] == background)
      & (df["Specialty"] == specialty)
    ]
  # for each (group, pack), total score
  df_group_pack = (df
    .groupby(groupby_cols + [pack_index_col])[score_col]
    .sum()
    .reset_index())
  # for each pack, average score per group
  df_pack_map = (df_group_pack
    .groupby(pack_index_col)[score_col]
    .mean()
    .to_dict())
  # for each (group, pack), add average score per group based on pack
  df_group_pack["pack_avg"] = df_group_pack[pack_index_col].map(df_pack_map)
  # for each group, total score and average score over all its packs
  df_group = (df_group_pack
    .groupby(groupby_cols)[[score_col, "pack_avg"]]
    .sum()
    .reset_index())
  # compare group's score to average group's score
  df_group["score"] = df_group[score_col] / df_group["pack_avg"]
  return df_group.sort_values("score", ascending=False)

In [135]:
# make Background and Specialty columns look nice
df_decks = (df_decks.rename(columns={
  "background": "Background",
  "specialty": "Specialty",
}))
for col in ["Background", "Specialty"]:
  df_decks[col] = df_decks[col].apply(
    lambda s: s.replace('_', ' ').title()
  )

# initialize tables for export
tables = {}
for background in df_decks["Background"].unique():
  for specialty in df_decks["Specialty"].unique():
    bg_sp = f'{background}+{specialty}'
    tables[bg_sp] = {}

In [136]:
# score Background + Specialty combinations
df_background_specialty = group_and_score(df_decks,
  groupby_cols=["Background", "Specialty"],
  score_col="group_score",
  pack_index_col="deck_pack_index")
df_background_specialty = (df_background_specialty
  .drop(columns=["group_score", "pack_avg"])
  .rename(columns={"score": "Score"}
))
# save to export
tables['background_specialty'] = df_background_specialty.to_dict(orient='records')

print('Background + Specialty combination scores:')
df_background_specialty

Background + Specialty combination scores:


,Background,Specialty,Score
11,Shepherd,Conciliator,2.699273
22,Traveler,Explorer,2.288612
0,Artisan,Artificer,2.111409
18,Talespinner,Shaper,1.880108
23,Traveler,Shaper,1.434186
14,Shepherd,Spirit Speaker,1.405698
8,Forager,Shaper,1.403209
19,Talespinner,Spirit Speaker,1.327828
16,Talespinner,Conciliator,1.131234
7,Forager,Explorer,0.954841


Now we look at the cards chosen by decks in a specific Background and Specialty combination. We use the same function as above. We extracted `role_id` for each deck above. We want to score all versions of a card together, so we can either group by its `code` or `name`.

In [137]:
for background in df_background_specialty["Background"].unique():
  for specialty in df_background_specialty["Specialty"].unique():
    # we groupby code/role, not role_id
    df_decks["role"] = df_decks["role_id"].map(card_id_to_name)
    role_scores = group_and_score(
      df=df_decks,
      groupby_cols=["role"],
      score_col="group_score",
      pack_index_col="deck_pack_index",
      background=background,
      specialty=specialty
    )

    role_scores = (role_scores
      .drop(columns=["group_score", "pack_avg"])
      .rename(columns={
        "role": "Role",
        "score": "Score"
      }
    ))
    bg_sp = f'{background}+{specialty}'
    tables[bg_sp]['role'] = role_scores.to_dict(orient='records')

tables['Talespinner+Spirit Speaker']['role']

[{'Role': 'Keeper of the Grove', 'Score': 1.4597123366994063},
 {'Role': 'Cartographer of Many Worlds', 'Score': 0.9735440683483554},
 {'Role': 'Descendant of the Guide', 'Score': 0.5667435949522385}]

In [138]:
for background in df_background_specialty["Background"].unique():
  for specialty in df_background_specialty["Specialty"].unique():
    df_decks["equip_sum"] = df_decks["equip_sum"].astype(int)
    equip_scores = group_and_score(df_decks,
      groupby_cols=["equip_sum"],
      score_col="group_score",
      pack_index_col="deck_pack_index",
      background=background,
      specialty=specialty
    )
    equip_scores = (equip_scores
      .drop(columns=["group_score", "pack_avg"])
      .rename(columns={
        "equip_sum": "Total Equip",
        "score": "Score"
      }
    ))
    bg_sp = f'{background}+{specialty}'
    tables[bg_sp]['equip'] = equip_scores.to_dict(orient='records')

tables['Talespinner+Spirit Speaker']['equip']

[{'Total Equip': 5, 'Score': 3.49528285563689},
 {'Total Equip': 4, 'Score': 2.353773138740833},
 {'Total Equip': 2, 'Score': 1.2821035553398579},
 {'Total Equip': 3, 'Score': 0.7238655000377012},
 {'Total Equip': 6, 'Score': 0.6301992918971288},
 {'Total Equip': 7, 'Score': 0.24143201961094532},
 {'Total Equip': 1, 'Score': 0.12764647650279407},
 {'Total Equip': 0, 'Score': 0.11378554310815124},
 {'Total Equip': 9, 'Score': 0.031911619125698516}]

In [139]:
def score_aspect(background, specialty, aspect):
  aspect_scores = group_and_score(df_decks,
    groupby_cols=[aspect],
    score_col="group_score",
    pack_index_col="deck_pack_index",
    background=background,
    specialty=specialty
  )
  return aspect_scores

asp_to_aspect_lookup = {
  'awa': 'Awareness',
  'fit': 'Fitness',
  'foc': 'Focus',
  'spi': 'Spirit'
}

for background in df_background_specialty["Background"].unique():
  for specialty in df_background_specialty["Specialty"].unique():
    for asp, aspect in asp_to_aspect_lookup.items():
      df_aspect = score_aspect(background, specialty, asp)
      df_aspect = (df_aspect
        .drop(columns=["group_score", "pack_avg"])
        .rename(columns={
          asp: aspect,
          "score": "Score"
        }
      ))
      bg_sp = f'{background}+{specialty}'
      tables[bg_sp][aspect] = df_aspect.to_dict(orient='records')

tables['Talespinner+Spirit Speaker']['Awareness']

[{'Awareness': 1, 'Score': 2.707205570211743},
 {'Awareness': 2, 'Score': 0.8592603349592008},
 {'Awareness': 3, 'Score': 0.38296274233654526},
 {'Awareness': 4, 'Score': 0.05057135249251167}]

In [140]:
def extract_cards(background, specialty):
  score_col = "group_score"
  pack_index_col = "deck_pack_index"

  df = df_decks[
    (df_decks["Background"] == background)
    & (df_decks["Specialty"] == specialty)
    & (df_decks[score_col] > 0)
  ]

  # attach score and pack_index of deck to each card
  bg_sp_slots_data = []
  for _, deck in df.iterrows():
    slots = deck["slots"]
    for card_id, _ in slots.items():
      card_name = card_id_to_name[card_id]
      bg_sp_slots_data.append({
        "name": card_name,
        score_col: deck[score_col],
        pack_index_col: deck[pack_index_col]
      })

  # we don't need individual cards, just their sum
  bg_sp_slots_long = pd.DataFrame(bg_sp_slots_data)
  bg_sp_slots = (
    bg_sp_slots_long.groupby(["name", pack_index_col])
    [score_col].sum()
    .reset_index()
    .astype({pack_index_col: "Int64", score_col: "float64"})
  )

  # bg_sp_slots may not contain all allowed outside interest cards
  # get all cards that users can build their decks with
  available_cards = (df_cards[
    df_cards["supergroup_id"].isin(['Personality', 'Background', 'Specialty'])
      # & df_cards["is_latest"]
    ][["name", "supergroup_id", "subgroup_id"]]
    .drop_duplicates()
    .copy())
  # distinguish outside interest cards from background and specialty cards
  outside_interest_mask = (
    (available_cards["supergroup_id"] == 'Background')
      & (available_cards["subgroup_id"] != background)
    ) | (
    (available_cards["supergroup_id"] == 'Specialty')
      & (available_cards["subgroup_id"] != specialty))
  available_cards.loc[outside_interest_mask, "supergroup_id"] = 'Outside'
  available_cards.loc[outside_interest_mask, "subgroup_id"] = 'Interest'

  # join all cards with card data from decks
  bg_sp_cards = (available_cards
    .merge(
      bg_sp_slots,
      how='left',
      on="name")
    .astype({pack_index_col: "Int64", score_col: "float64"})
    .fillna(0)
  )
  return bg_sp_cards

In [141]:
def score_supergroup_subgroup(df, supergroup_id, subgroup_id):
  df = df[
    (df["supergroup_id"] == supergroup_id)
    & (df["subgroup_id"] == subgroup_id)
  ]
  group_scores = group_and_score(df,
    groupby_cols=["name"],
    score_col="group_score",
    pack_index_col="deck_pack_index"
  )
  return group_scores

df_card_lookup = df_cards[[
  "name",
  "equip",
  "level",
  "aspect_id"]].drop_duplicates()
assert df_card_lookup["name"].is_unique
df_card_lookup["equip"] = df_card_lookup["equip"].astype('Int64')
df_card_lookup["aspect_level"] = df_card_lookup.apply(
  lambda row: None if pd.isna(row["level"]) else f'{row["aspect_id"]} {int(row["level"])}',
  axis=1
)

for background in df_background_specialty["Background"].unique():
  for specialty in df_background_specialty["Specialty"].unique():
    bg_sp_cards = extract_cards(background, specialty)
    for supergroup_id, subgroup_id in [
        ('Personality', 'Awareness'),
        ('Personality', 'Fitness'),
        ('Personality', 'Focus'),
        ('Personality', 'Spirit'),
        ('Background', background),
        ('Specialty', specialty),
        ('Outside', 'Interest')]:
      df_group = score_supergroup_subgroup(bg_sp_cards, supergroup_id, subgroup_id)
      df_group = df_group.drop(columns=[
        "group_score",
        "pack_avg"
      ])
      if supergroup_id == 'Outside' and subgroup_id == 'Interest':
        df_group = df_group[df_group["score"] > 1]

      if supergroup_id == 'Personality':
        df_group = (df_group
          .merge(
            df_card_lookup[["name", "aspect_level"]],
            how="left",
            on="name"
          )
          .rename(columns={
            "name": "Name",
            "aspect_level": "Aspect requirement",
            "score": "Score"
          })
        )
        group_name = f'{supergroup_id}-{subgroup_id}'
      else:
        df_group = (df_group
          .merge(
            df_card_lookup[["name", "aspect_level", "equip"]],
            how="left",
            on="name"
          )
          .rename(columns={
            "name": "Name",
            "equip": "Equip value",
            "aspect_level": "Aspect requirement",
            "score": "Score"
          })
        )
        if supergroup_id == 'Outside' and subgroup_id == 'Interest':
          group_name = 'Outside Interest'
        else:
          group_name = supergroup_id

      bg_sp = f'{background}+{specialty}'
      tables[bg_sp][group_name] = df_group.to_dict(orient='records')
      print(f'\n{group_name}')
      print(df_group)


Personality-Awareness
         Name     Score Aspect requirement
0  Perceptive  1.971554              AWA 1
1  Insightful  1.094524              AWA 1
2    Thorough  0.890942              AWA 2
3   Dauntless  0.328018              AWA 1
4   Observant  0.243295              AWA 3
5    Vigilant  0.176897              AWA 1

Personality-Fitness
           Name     Score Aspect requirement
0          Bold  1.609393              FIT 2
1    Passionate  1.369809              FIT 1
2      Balanced  0.827828              FIT 1
3  Enthusiastic  0.818754              FIT 2
4     Steadfast  0.481693              FIT 3
5    Determined  0.258543              FIT 1

Personality-Focus
         Name     Score Aspect requirement
0   Versatile  1.474223              FOC 1
1  Deliberate  1.337577              FOC 1
2  Meticulous  1.225119              FOC 1
3      Astute  1.125115              FOC 2
4   Inventive  0.217729              FOC 1
5   Ingenious  0.212368              FOC 3

Personality-Spirit


In [142]:
with open('project/fifteen_good_cards.json', 'w') as f:
  json.dump(tables, f)

-- OLD STUFF --

For the actual deck, we labeled each card by `supergroup_id` and `subgroup_id` above. This splits available cards into seven groups, which we score separately: four aspects of Personality cards, their Background, their Specialty, and their Outside Interest.

In [ ]:
def score_background_specialty(background, specialty, score_col, pack_index_col):
  bg_sp_decks = df_decks[(df_decks["background"] == background)
    & (df_decks["specialty"] == specialty)
    & (df_decks[score_col] > 0)].copy()

  bg_sp_decks["role"] = bg_sp_decks["role_id"].map(card_id_to_name)
  print('Role scores:')
  group_and_score_wrapper(bg_sp_decks, "role", score_col, pack_index_col)

  bg_sp_cards = extract_cards(bg_sp_decks, score_col, pack_index_col)

  build_order = [('Personality', 'Awareness', 3),
                ('Personality', 'Fitness', 3),
                ('Personality', 'Focus', 3),
                ('Personality', 'Spirit', 3),
                ('Background', background.capitalize(), 10),
                ('Specialty', specialty.capitalize(), 10),
                ('Outside', 'Interest', 10)]

  for supergroup_id, subgroup_id, num_results in build_order:
    build_cards = (bg_sp_cards[
      (bg_sp_cards["supergroup_id"] == supergroup_id)
      & (bg_sp_cards["subgroup_id"] == subgroup_id)]
      .copy())
      
    print(f'\n{supergroup_id} {subgroup_id} scores:')
    group_and_score_wrapper(build_cards, "name", score_col, pack_index_col,
      num_results)

background = 'shepherd'
specialty = 'conciliator'
score_background_specialty(background, specialty,
  score_col="group_score",
  pack_index_col="deck_pack_index")

Role scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Voice of the Elders                 497.2  / 401.7  = 1.238 
Guardian                            314.6  / 401.7  = 0.783 
Pathwarden                          19.7   / 28.0   = 0.702 

Personality Awareness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Perceptive                          398.4  / 200.9  = 1.983 
Insightful                          212.3  / 200.9  = 1.057 
Thorough                            175.7  / 200.9  = 0.875 

Personality Fitness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Bold                                317.9  / 200.9  = 1.583 
Passionate                          277.2  / 200.9  = 1.380 
Balanced                            167.2  / 200.9  = 0.833 

Personality

In [125]:
background = 'traveler'
specialty = 'artificer'
score_background_specialty(background, specialty,
  score_col="group_score",
  pack_index_col="deck_pack_index")

Role scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Exceptional Tinkerer                125.8  / 74.3   = 1.695 
Gregarious Inventor                 4.6    / 7.7    = 0.593 
Masterful Engineer                  25.8   / 74.3   = 0.347 

Personality Awareness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Thorough                            64.9   / 32.4   = 2.004 
Perceptive                          53.8   / 32.4   = 1.660 
Insightful                          19.8   / 32.4   = 0.612 

Personality Fitness scores:
Name                                Group  / Avg    = Score 
------------------------------------------------------------
Balanced                            50.4   / 37.1   = 1.358 
Passionate                          48.8   / 37.1   = 1.316 
Steadfast                           4.1    / 3.9    = 1.061 

Personality

In [22]:
df_decks.columns

Index(['id', 'user_id', 'slots', 'side_slots', 'extra_slots', 'version',
       'name', 'description', 'awa', 'spi', 'fit', 'foc', 'created_at',
       'updated_at', 'meta', 'user', 'taboo_set_id', 'published',
       'previous_deck', 'next_deck', 'copy_count', 'comment_count',
       'like_count', 'liked_by_user', 'original_deck', 'campaign', 'comments',
       'background', 'specialty', 'role_id', 'is_ignore', 'deck_pack_index',
       'has_15_cards', 'has_spoiler', 'is_latest', 'all_decks_parent_id',
       'parent_id', 'inspiration_id', 'user_weight', 'deck_weight',
       'taboo_weight', 'group_score'],
      dtype='object')

In [27]:
for aspect in ['awa', 'fit', 'foc', 'spi']:
  sums = (
    df_decks.groupby(["background", "specialty", aspect], as_index=False)["deck_weight"]
    .sum()
  )
  top = (
    sums.sort_values("deck_weight", ascending=False)
    .drop_duplicates(["background", "specialty"])
  )
  pivot = top.pivot(index="background", columns="specialty", values=aspect)
  print(f"\nMost common {aspect}:")
  print(pivot)


Most common awa:
specialty    artificer  conciliator  explorer  shaper  spirit_speaker
background                                                           
artisan              2            2         2       2               2
forager              2            2         2       3               1
shepherd             2            2         2       2               1
talespinner          2            2         1       1               1
traveler             2            2         2       2               1

Most common fit:
specialty    artificer  conciliator  explorer  shaper  spirit_speaker
background                                                           
artisan              1            2         2       2               2
forager              2            2         2       2               2
shepherd             2            2         2       2               2
talespinner          1            1         2       2               2
traveler             2            2         3       3 